In [3]:
import requests
import pandas as pd

from keys import TMDB_API_KEY

BASE_URL = "https://api.themoviedb.org/3"



In [ ]:
def search_movie(term):
    url = f"{BASE_URL}/search/movie"
    params = {
        "api_key": TMDB_API_KEY,
        "query": term
    }
    response = requests.get(url, params=params)

    # Check to make sure results are actually being sent
    if response.status_code != 200:
        print("Error: ", response.status_code)
        print(response.text)
        return []


    data = response.json()

    # If something is returned that isn't expected
    if "results" not in data or data["results"] is None:
        print("TMDB returned no results.")
        print(data)
        return []
    results = data.get("results", [])

    movies = []
    for m in results:
        movies.append({
            "id": m["id"],
            "title": m["title"],
            "year": (m.get("release_date") or "")[:4],
            "overview": m.get("overview", ""),
            "genre_ids": m.get("genre_ids", [])
        })



    return movies

In [ ]:
# Testing search_movie with hardcoded search term
movies = search_movie("Idiocracy")
for i, m in enumerate(movies, start=1):
    print(i, m["id"], m["title"], m["year"])

1 7512 Idiocracy 2006


In [ ]:
def submit_review(movie_id, review, genre_ids=None):
    # Validate User review
    if review < 1 or review > 5:
        print("You must pick between 1 and 5 stars.")
        return
    
    # Load existing reviews (if any)
    try:
        df = pd.read_csv("reviews.csv")
    except FileNotFoundError:
        df = pd.DataFrame(columns=["movie_id", "review", "genre_ids"])

    # Add a new row for the new review
    new_row = {
        "movie_id": movie_id,
        "review": review,
        "genre_ids": str(genre_ids) if genre_ids is not None else "" # Add empty string if no genre id string to keep csv clean
    }
    # Add new review to the bottom of reviews.csv
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

    # Save new review to reviews.csv
    df.to_csv("reviews.csv", index=False)
    print("Your review was saved!")

In [9]:
# Test submit_review 
first = movies[0]
submit_review(first["id"], 5, first["genre_ids"])

Your review was saved!
